# RNAPhaseek — score and design protein-free LLPS RNAs

**One-click Colab for the RNAPhaseek production model** ([HF Hub](https://huggingface.co/), [GitHub](https://github.com/)).

RNAPhaseek predicts the probability that an RNA undergoes **protein-free liquid–liquid phase separation (LLPS)**.

**What this notebook does:**
- **Score** any RNA sequence(s) you paste, getting P(LLPS) per sequence.
- **Design** new high-scoring RNAs de novo with a GA or DEN generator.

### Runtime
GPU is recommended but not required. *Runtime → Change runtime type → T4 GPU* for ~5× speed.

### Customize
Edit `GITHUB_REPO` and `HF_REPO_ID` in the Setup cell below to point at your fork / release. The defaults assume the public release.

## 1. Setup (run once per session — ~3 min)

Installs dependencies, clones the repo, and downloads the model weights from Hugging Face Hub.

In [ ]:
#@title Setup { display-mode: "form" }
#@markdown Edit these if you forked the project / re-uploaded the weights.
GITHUB_REPO  = "QuercusCode/RNAPhaseek"   #@param {type:"string"}
HF_REPO_ID   = "YOUR-HF-USER/rnaphaseek"       #@param {type:"string"}

import os, sys, subprocess, pathlib

# --- ViennaRNA (needed by the biophysical-feature extractor) ---
if not pathlib.Path("/usr/local/condarc").exists() and not os.path.exists("/opt/conda/bin/mamba"):
    print("[setup] installing condacolab + viennarna (slowest step, ~2 min)")
    !pip install -q condacolab
    import condacolab; condacolab.install_mambaforge()
    # condacolab.install_mambaforge() restarts the runtime. After restart, re-run this cell.
else:
    print("[setup] conda already present")

try:
    import RNA; print(f"[setup] ViennaRNA already importable, v={RNA.__version__ if hasattr(RNA,'__version__') else 'unknown'}")
except ImportError:
    print("[setup] installing ViennaRNA via mamba")
    !mamba install -c bioconda -y viennarna >/dev/null 2>&1 || conda install -c bioconda -y viennarna
    import RNA; print("[setup] ViennaRNA OK")

# --- Project code + python deps ---
if not pathlib.Path("RNAPhaseek").exists():
    print(f"[setup] cloning {GITHUB_REPO}")
    !git clone -q https://github.com/{GITHUB_REPO}.git RNAPhaseek
%cd RNAPhaseek

print("[setup] installing python deps")
!pip install -q multimolecule "transformers>=4.40" "torch>=2.1" biopython huggingface_hub tqdm scikit-learn matplotlib pandas

# --- Weights from HF Hub ---
from huggingface_hub import hf_hub_download
print(f"[setup] downloading model weights from {HF_REPO_ID}")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="final_model.pt")
NORM_PATH  = hf_hub_download(repo_id=HF_REPO_ID, filename="norm_stats.npz")

# Place them where rnaphaseek.py expects them (so its default paths just work)
os.makedirs("model/strict_eval_v13_production", exist_ok=True)
for src, dst in [(MODEL_PATH, "model/strict_eval_v13_production/final_model.pt"),
                 (NORM_PATH,  "model/strict_eval_v13_production/norm_stats.npz")]:
    if not pathlib.Path(dst).exists():
        os.symlink(src, dst)

print("[setup] OK — proceed to the next cell.")

## 2. Load the scorer (run once)

In [ ]:
import sys
sys.path.insert(0, ".")
from rnaphaseek import RNAPhaseekScorer, read_fasta
import torch, io, csv, numpy as np

scorer = RNAPhaseekScorer(
    model_path="model/strict_eval_v13_production/final_model.pt",
    norm_path="model/strict_eval_v13_production/norm_stats.npz",
    quiet=False,
)
print(f"\nRNAPhaseek ready on {scorer.device}")

## 3. Score your RNA sequences

Paste a FASTA below (one or many sequences). Press the ▶ button to score them.

In [ ]:
#@title Score sequences { display-mode: "form" }
input_fasta = ">seq_g4_repeat\nGGGAGGGAGGGAGGGUUUUUUUUUUUUUUUUUUUUU\n>seq_au_rich\nUUUAAAUUUAAAUUUAAAUUUAAAUUUAAAUUUAAA\n>seq_random\nACGUACGUACGUACGUACGUACGUACGUACGUACGU"  #@param {type:"string"}
threshold   = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.01}
save_csv    = True  #@param {type:"boolean"}

# --- Parse ---
with open("/tmp/input.fasta", "w") as f:
    f.write(input_fasta.strip() + "\n")
recs = read_fasta("/tmp/input.fasta")
seqs = [RNAPhaseekScorer._norm_seq(s) for _, s in recs]
if not seqs:
    raise SystemExit("No sequences parsed from the input. Did you forget the '>' header line?")
print(f"Scoring {len(seqs)} sequence(s) ...")

# --- Score ---
probs = scorer.score(seqs)

# --- Output table ---
rows = [("id", "length", "GC_percent", "P_LLPS", f"call@{threshold:.2f}")]
for (h, _), s, p in zip(recs, seqs, probs):
    gc = 100 * sum(c in "GC" for c in s) / max(len(s), 1)
    rows.append((h.split()[0], len(s), f"{gc:.1f}", f"{p:.4f}", "LLPS" if p >= threshold else "no"))

import pandas as pd
df = pd.DataFrame(rows[1:], columns=rows[0])
print()
print(df.to_string(index=False))

if save_csv:
    df.to_csv("scores.csv", index=False)
    from google.colab import files
    files.download("scores.csv")
    print("\nscores.csv downloaded to your machine.")

## 4. Design — GA generator (~30–60 s)

Genetic algorithm optimizing model P(LLPS). Returns the top *N* designs.

In [ ]:
#@title GA design { display-mode: "form" }
length       = 100   #@param {type:"slider", min:30, max:1000, step:10}
n_top        = 5     #@param {type:"slider", min:1, max:30, step:1}
generations  = 25    #@param {type:"slider", min:5, max:80, step:5}
seed         = 0     #@param {type:"integer"}
download_fasta = True  #@param {type:"boolean"}

import random, numpy as np

rng = random.Random(seed)
BASES = "ACGU"
POP, ELITE, MUT, TOURN = 64, 10, 0.04, 3
pop = ["".join(rng.choice(BASES) for _ in range(length)) for _ in range(POP)]
cache = {}
for g in range(generations):
    new = [s for s in pop if s not in cache]
    if new:
        scored = scorer.score(new)
        for s, p in zip(new, scored): cache[s] = float(p)
    fit = [cache[s] for s in pop]
    order = sorted(range(POP), key=lambda i: -fit[i])
    if (g + 1) % 5 == 0 or g == 0:
        print(f"  gen {g+1:>2}/{generations}: best={fit[order[0]]:.4f}  mean={np.mean(fit):.4f}")
    elites = [pop[i] for i in order[:ELITE]]; newpop = list(elites)
    def tour():
        c = rng.sample(range(POP), TOURN); return pop[max(c, key=lambda i: fit[i])]
    while len(newpop) < POP:
        x, y = tour(), tour(); i, j = sorted(rng.sample(range(1, length), 2))
        child = x[:i] + y[i:j] + x[j:]
        child = "".join(rng.choice(BASES) if rng.random() < MUT else ch for ch in child)
        newpop.append(child)
    pop = newpop

top = sorted(set(pop), key=lambda s: -cache.get(s, 0))[:n_top]
print(f"\nTop {n_top} GA designs:")
lines = []
for i, s in enumerate(top):
    h = f">ga_design_{i}_P{cache[s]:.3f}"
    print(f"  {h}\n  {s}")
    lines.append(h); lines.append(s)

if download_fasta:
    open("designs_ga.fasta", "w").write("\n".join(lines) + "\n")
    from google.colab import files; files.download("designs_ga.fasta")
    print("\ndesigns_ga.fasta downloaded.")

## 5. Design — DEN generator (diverse library, ~2–5 min)

Trains a small generator network to output a **diverse** batch of high-P(LLPS) sequences. Slower than the GA but the designs are mutually distinct (low pairwise identity), good for library generation.

In [ ]:
#@title DEN design { display-mode: "form" }
length      = 200  #@param {type:"slider", min:50, max:1000, step:10}
n_top       = 10   #@param {type:"slider", min:5, max:50, step:1}
steps       = 200  #@param {type:"slider", min:50, max:600, step:50}
seed        = 0    #@param {type:"integer"}
download_fasta = True  #@param {type:"boolean"}

import subprocess, sys, json, pathlib
out_fa = f"designs_den_{length}nt.fasta"
cmd = [sys.executable, "scripts/generation/den_design_v6.py",
       "--length", str(length), "--n", str(n_top),
       "--steps", str(steps), "--seed", str(seed), "--out", out_fa]
print(" ".join(cmd) + "\n")
subprocess.run(cmd, check=True)

print(f"\nDesigns written to {out_fa}:")
print(open(out_fa).read())

if download_fasta:
    from google.colab import files; files.download(out_fa)
    print(f"{out_fa} downloaded.")

## Notes

- **1022-nt context limit** (RNA-FM backbone). Sequences longer than that get scored on their first 1022 nt — a held-out study showed no accuracy loss vs full-length scoring.
- **Threshold**: default 0.5 (F1=0.860 on CV). For stricter calibration use 0.63 (F1-optimal, F1=0.863).
- **Trustworthiness check**: a design with very high P(LLPS) on a low-complexity sequence may be a composition artefact rather than a structure-driven signal. The full CLI's `validate` subcommand compares against scrambles; for a quick check, score a di-shuffled version of your sequence — a true structure-driven hit will lose most of its score.
- **Issues / questions**: open one on the [GitHub repo](https://github.com/).